# Understanding Data Leakage Through Simple Examples

Data leakage is one of the most dangerous and misunderstood problems in machine learning.
It often produces impressive validation scores and then fails in production.

This notebook builds an intuitive and practical understanding of leakage using simple examples.

## Learning Goals

By the end of this lesson, you should be able to:
- Define data leakage clearly
- Recognize target, temporal, and contamination leakage
- Apply the prediction moment test
- Build leakage-safe splitting and preprocessing workflows

## What Is Data Leakage?

Data leakage occurs when a model uses information during training that would not be available at prediction time in the real world.

Core rule:

A model must only learn from information that exists at the moment a real-world prediction is made.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Why Leakage Is Dangerous

Leakage is dangerous because it:
- Produces artificially high metrics
- Hides generalization failure
- Misleads decision makers
- Wastes engineering time
- Damages trust after deployment

Leakage does not make training look worse.
It makes evaluation dishonest.

## Example 1: Target Leakage

Scenario: predict customer churn.

Feature mistake: using `CancellationReason`.

`CancellationReason` exists after churn, so it leaks outcome information.

In [ ]:
rng = np.random.default_rng(42)
n = 1000

tenure = rng.integers(1, 60, n)
monthly_charges = rng.normal(70, 15, n).clip(20, 150)
support_calls = rng.poisson(2, n)

churn_prob = 1 / (1 + np.exp((tenure - 18) / 5))
churn_prob = np.clip(churn_prob + 0.05 * (support_calls > 3), 0, 1)
churn = (rng.random(n) < churn_prob).astype(int)

df = pd.DataFrame({
    "Tenure": tenure,
    "MonthlyCharges": monthly_charges,
    "SupportCallsLast90Days": support_calls,
    "Churn": churn,
})

# Leaky feature: appears only when churn already happened
df["CancellationReason"] = np.where(df["Churn"] == 1, "Price", None)

df.head()

In [ ]:
# Wrong: includes leaky feature
X_leaky = df[["Tenure", "MonthlyCharges", "SupportCallsLast90Days", "CancellationReason"]]
y = df["Churn"]

num_cols = ["Tenure", "MonthlyCharges", "SupportCallsLast90Days"]
cat_cols = ["CancellationReason"]

pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

model = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=1000)),
])

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Leaky test accuracy:", round(accuracy_score(y_test, pred), 4))

In [ ]:
# Correct: remove leaky feature
X_clean = df[["Tenure", "MonthlyCharges", "SupportCallsLast90Days"]]

clean_model = Pipeline([
    ("pre", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

X_train, X_test, y_train, y_test = train_test_split(X_clean, y, test_size=0.2, random_state=42, stratify=y)
clean_model.fit(X_train, y_train)
pred = clean_model.predict(X_test)

print("Leakage-safe test accuracy:", round(accuracy_score(y_test, pred), 4))

## Example 2: Temporal Leakage

If a feature reflects events after the prediction moment, it leaks future information.

Simple rule:
- Predict at time $t$
- Use only information available at or before time $t$

In [ ]:
dates = pd.date_range("2024-01-01", periods=12, freq="M")
ts_df = pd.DataFrame({
    "Date": dates,
    "FeatureAtPredictionTime": np.linspace(10, 22, 12),
    "FutureSignal": np.linspace(100, 200, 12),  # pretend this appears later
    "Target": [0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1],
})

# Correct split: chronological
split_idx = int(len(ts_df) * 0.8)
train_ts = ts_df.iloc[:split_idx]
test_ts = ts_df.iloc[split_idx:]

print("Train end date:", train_ts["Date"].max().date())
print("Test start date:", test_ts["Date"].min().date())

## Example 3: Train-Test Contamination

Wrong order:
1. Fit scaler on full data
2. Split train/test

Correct order:
1. Split train/test
2. Fit scaler on training data only
3. Transform test with training-fitted scaler

In [ ]:
X = df[["Tenure", "MonthlyCharges", "SupportCallsLast90Days"]]
y = df["Churn"]

# Correct contamination-safe split/fit sequence
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_scaled, y_train)
pred = clf.predict(X_test_scaled)

print("Accuracy (contamination-safe):", round(accuracy_score(y_test, pred), 4))

## Example 4: Feature Derived from Target

If a feature would disappear when the target is unknown, it is likely target-derived leakage.

Example:
- Target: `Defaulted`
- Leaky feature: `DaysUntilDefault`

`DaysUntilDefault` is defined by the outcome itself.

## Example 5: Aggregation Leakage

If you compute statistics using the full dataset before splitting, test labels can leak into features.

Wrong:
- Compute `RegionalChurnRate` on all rows
- Then split

Right:
- Split first
- Compute aggregation on training rows only
- Map to test rows using train-only statistics

## Prediction Moment Test

Use this checklist for every feature:
1. At prediction time, does this value already exist?
2. Could this value be influenced by the target outcome?
3. Could this value include information from future rows?

If any answer is uncertain, investigate before training.

In [ ]:
def prediction_moment_test(feature_name: str, available_at_prediction_time: bool, target_dependent: bool, future_dependent: bool) -> str:
    if not available_at_prediction_time:
        return f"{feature_name}: REJECT (not available at prediction time)"
    if target_dependent:
        return f"{feature_name}: REJECT (target leakage risk)"
    if future_dependent:
        return f"{feature_name}: REJECT (temporal leakage risk)"
    return f"{feature_name}: ACCEPT"

print(prediction_moment_test("CancellationReason", False, True, False))
print(prediction_moment_test("SupportCallsLast90Days", True, False, False))

## Leakage-Safe Practice Rules

- Separate `X` and `y` early
- Split before any fit operation
- Fit transformations on training data only
- Avoid target-derived and post-outcome features
- Use chronological split for time-dependent data
- Document feature availability assumptions

## Closing Reflection

Data leakage is usually a thinking error, not a syntax error.

Define clearly:
- What you are predicting
- When prediction is made
- What information exists at that moment

If the model sees information it would not have in real life, evaluation is invalid.